# Problema 1: El reporte demasiado bueno
**Alumno:** Owen Adiel Solis Zatarain  
**Expediente / Semilla:** `223201053` ($a = 3$, $b = 5$)

---

## (a) Reproducir
Abajo está el **código original intacto de la consultora**. Según las instrucciones del examen, modifícalo tú mismo para tu versión personal:
1. Cambia `k=15` por $k = 10 + a = 13$.
2. Cambia `random_state=0` por tu semilla (`223201053`) en todos lados.
3. Como $b = 5 \implies b \bmod 3 = 2$, cambia el `RandomForestClassifier` por `LogisticRegression(C=c, max_iter=2000)` probando $c \in \{0.01, 0.1, 1, 10\}$.

In [17]:
from sklearn.linear_model import LogisticRegression
import io, zipfile, urllib.request
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
externo = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
interno = zipfile.ZipFile(io.BytesIO(externo.read("bank.zip")))
datos = pd.read_csv(interno.open("bank-full.csv"), sep=";")
datos_crudos = datos.copy()

y = (datos.pop("y") == "yes").astype(int)
X = pd.get_dummies(datos, drop_first=True).astype(float)

# Las clases están desbalanceadas: repetimos los positivos para equilibrarlas
pos = y == 1
X = pd.concat([X] + [X[pos]] * 6, ignore_index=True)
y = pd.concat([y] + [y[pos]] * 6, ignore_index=True)

X = StandardScaler().fit_transform(X)
X = SelectKBest(f_classif, k=13).fit_transform(X, y)

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=223201053)

mejor = None
for c in [0.01,0.1,1,10]:
    modelo = LogisticRegression(
        C=c, max_iter=2000, random_state=223201053
    )
    modelo.fit(X_ent, y_ent)
    auc = roc_auc_score(y_pru, modelo.predict_proba(X_pru)[:, 1])
    if mejor is None or auc > mejor[0]:
        mejor = (auc, c, accuracy_score(y_pru, modelo.predict(X_pru)))

print(f"Profundidad elegida: {mejor[1]}")
print(f"Exactitud en prueba: {mejor[2]:.3f} AUC en prueba: {mejor[0]:.3f}")

Profundidad elegida: 10
Exactitud en prueba: 0.807 AUC en prueba: 0.893


---
## (b) Encontrar los problemas
Haz una tabla con todos los problemas que encuentres en el código (hay más de tres):

| # | Problema en el código | Tipo de problema | ¿Por qué infla o distorsiona el resultado? |
| :-: | :--- | :--- | :--- |
| 1 | Incluir duration y variables de contacto| Fuga de información temporal (target leakage) | No se conoce antes de hacer la llamada y descarta llamadas cortas. |
| 2 | Repetir positivos | Fuga de información + distorsión de la prueba | Mete clones del entrenamiento en la prueba y cambia la proporción de positivos de 11.7% a 48.1%|
| 3 | Usar todo el dataset con StandardScaler().fit_transform(x) | Fuga de información en preprocesamiento | Usa la media y desviación estándar de la prueba |
| 4 | SelectKBest(...).fit_transform(X, y) con todo el dataset | Fuga de información supervisada | elige las 13 variables usando también las etiquetas y de la prueba. |
| 5 | train_test_split aleatorio en vez de cronológico | Mala práctica de validación temporal | Mezcla llamadas de 2008 a 2010 y entrena con el futuro para predecir el pasado |
| 6 | Elegir el mejor hiperparámetro (C) usando X_pru, y_pru | Mala práctica de validación | Usa el conjunto de prueba para seleccionar el modelo y reportar ese mismo puntaje |

---
## (c) Medir cada uno
Corrige los problemas **uno a la vez** (dejando los demás como están) y después **todos juntos**. Presenta una tabla con la exactitud y el AUC de cada variante.

In [18]:
datos = datos_crudos.copy()
datos = datos.drop(columns=["duration"])
y = (datos.pop("y") == "yes").astype(int)
X = pd.get_dummies(datos, drop_first=True).astype(float)

# Las clases están desbalanceadas: repetimos los positivos para equilibrarlas
pos = y == 1
X = pd.concat([X] + [X[pos]] * 6, ignore_index=True)
y = pd.concat([y] + [y[pos]] * 6, ignore_index=True)

X = StandardScaler().fit_transform(X)
X = SelectKBest(f_classif, k=13).fit_transform(X, y)

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=223201053)

mejor = None
for c in [0.01,0.1,1,10]:
    modelo = LogisticRegression(
        C=c, max_iter=2000, random_state=223201053
    )
    modelo.fit(X_ent, y_ent)
    auc = roc_auc_score(y_pru, modelo.predict_proba(X_pru)[:, 1])
    if mejor is None or auc > mejor[0]:
        mejor = (auc, c, accuracy_score(y_pru, modelo.predict(X_pru)))

print(f"Profundidad elegida: {mejor[1]}")
print(f"Exactitud en prueba: {mejor[2]:.3f} AUC en prueba: {mejor[0]:.3f}")

Profundidad elegida: 0.1
Exactitud en prueba: 0.689 AUC en prueba: 0.750


In [19]:
datos = datos_crudos.copy()
y = (datos.pop("y") == "yes").astype(int)
X = pd.get_dummies(datos, drop_first=True).astype(float)

X = StandardScaler().fit_transform(X)
X = SelectKBest(f_classif, k=13).fit_transform(X, y)

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=223201053)

mejor = None
for c in [0.01,0.1,1,10]:
    modelo = LogisticRegression(
        C=c, max_iter=2000, random_state=223201053
    )
    modelo.fit(X_ent, y_ent)
    auc = roc_auc_score(y_pru, modelo.predict_proba(X_pru)[:, 1])
    if mejor is None or auc > mejor[0]:
        mejor = (auc, c, accuracy_score(y_pru, modelo.predict(X_pru)))

print(f"Profundidad elegida: {mejor[1]}")
print(f"Exactitud en prueba: {mejor[2]:.3f} AUC en prueba: {mejor[0]:.3f}")

Profundidad elegida: 0.01
Exactitud en prueba: 0.898 AUC en prueba: 0.896


In [20]:
datos = datos_crudos.copy()
y = (datos.pop("y") == "yes").astype(int)
X = pd.get_dummies(datos, drop_first=True).astype(float)

# Las clases están desbalanceadas: repetimos los positivos para equilibrarlas
pos = y == 1
X = pd.concat([X] + [X[pos]] * 6, ignore_index=True)
y = pd.concat([y] + [y[pos]] * 6, ignore_index=True)

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=223201053)

scaler = StandardScaler()
X_ent = scaler.fit_transform(X_ent)
X_pru = scaler.transform(X_pru)

selector = SelectKBest(f_classif, k=13)
X_ent = selector.fit_transform(X_ent, y_ent)
X_pru = selector.transform(X_pru)

mejor = None
for c in [0.01,0.1,1,10]:
    modelo = LogisticRegression(
        C=c, max_iter=2000, random_state=223201053
    )
    modelo.fit(X_ent, y_ent)
    auc = roc_auc_score(y_pru, modelo.predict_proba(X_pru)[:, 1])
    if mejor is None or auc > mejor[0]:
        mejor = (auc, c, accuracy_score(y_pru, modelo.predict(X_pru)))

print(f"Profundidad elegida: {mejor[1]}")
print(f"Exactitud en prueba: {mejor[2]:.3f} AUC en prueba: {mejor[0]:.3f}")

Profundidad elegida: 10
Exactitud en prueba: 0.807 AUC en prueba: 0.893


In [21]:
datos = datos_crudos.copy()
y = (datos.pop("y") == "yes").astype(int)
X = pd.get_dummies(datos, drop_first=True).astype(float)

X = StandardScaler().fit_transform(X)
X = SelectKBest(f_classif, k=13).fit_transform(X, y)

# Partición cronológica (sin revolver fechas)
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, shuffle=False)

# Repetimos los positivos en el entrenamiento
pos = y_ent == 1
X_ent = np.concatenate([X_ent] + [X_ent[pos]] * 6, axis=0)
y_ent = np.concatenate([y_ent] + [y_ent[pos]] * 6, axis=0)

mejor = None
for c in [0.01, 0.1, 1, 10]:
    modelo = LogisticRegression(C=c, max_iter=2000, random_state=223201053)
    modelo.fit(X_ent, y_ent)
    auc = roc_auc_score(y_pru, modelo.predict_proba(X_pru)[:, 1])
    if mejor is None or auc > mejor[0]:
        mejor = (auc, c, accuracy_score(y_pru, modelo.predict(X_pru)))

print(f"C elegido: {mejor[1]}")
print(f"Exactitud en prueba: {mejor[2]:.3f} AUC en prueba: {mejor[0]:.3f}")

C elegido: 0.01
Exactitud en prueba: 0.750 AUC en prueba: 0.807


In [22]:
from sklearn.model_selection import cross_val_score

datos = datos_crudos.copy()
y = (datos.pop("y") == "yes").astype(int)
X = pd.get_dummies(datos, drop_first=True).astype(float)

pos = y == 1
X = pd.concat([X] + [X[pos]] * 6, ignore_index=True)
y = pd.concat([y] + [y[pos]] * 6, ignore_index=True)

X = StandardScaler().fit_transform(X)
X = SelectKBest(f_classif, k=13).fit_transform(X, y)

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=223201053)

mejor_cv = None
for c in [0.01, 0.1, 1, 10]:
    modelo = LogisticRegression(C=c, max_iter=2000, random_state=223201053)
    auc_cv = cross_val_score(modelo, X_ent, y_ent, cv=5, scoring="roc_auc").mean()
    if mejor_cv is None or auc_cv > mejor_cv[0]:
        mejor_cv = (auc_cv, c)

# Evaluamos UNA sola vez en la prueba con el C ganador del entrenamiento
modelo_final = LogisticRegression(C=mejor_cv[1], max_iter=2000, random_state=223201053)
modelo_final.fit(X_ent, y_ent)
acc_pru = accuracy_score(y_pru, modelo_final.predict(X_pru))
auc_pru = roc_auc_score(y_pru, modelo_final.predict_proba(X_pru)[:, 1])

print(f"C elegido por CV: {mejor_cv[1]}")
print(f"Exactitud en prueba: {acc_pru:.3f} AUC en prueba: {auc_pru:.3f}")

C elegido por CV: 10
Exactitud en prueba: 0.807 AUC en prueba: 0.893


In [23]:
datos = datos_crudos.copy()
datos = datos.drop(columns=["duration"])
y = (datos.pop("y") == "yes").astype(int)
X = pd.get_dummies(datos, drop_first=True).astype(float)

# 1. Partición cronológica primero (o aleatoria sin copias en prueba)
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, shuffle=False)

# 2. Escalar y seleccionar atributos SOLO con el entrenamiento
scaler = StandardScaler()
X_ent = scaler.fit_transform(X_ent)
X_pru = scaler.transform(X_pru)

selector = SelectKBest(f_classif, k=13)
X_ent = selector.fit_transform(X_ent, y_ent)
X_pru = selector.transform(X_pru)

# 3. Balancear clases SOLO en el entrenamiento
pos = y_ent == 1
X_ent = np.concatenate([X_ent] + [X_ent[pos].to_numpy()] * 6, axis=0) if hasattr(X_ent, "to_numpy") else np.concatenate([X_ent] + [X_ent[pos]] * 6, axis=0)
y_ent = np.concatenate([y_ent] + [y_ent[pos]] * 6, axis=0)

# 4. Elegir C con validación cruzada solo en entrenamiento
mejor_cv = None
for c in [0.01, 0.1, 1, 10]:
    modelo = LogisticRegression(C=c, max_iter=2000, random_state=223201053)
    auc_cv = cross_val_score(modelo, X_ent, y_ent, cv=5, scoring="roc_auc").mean()
    if mejor_cv is None or auc_cv > mejor_cv[0]:
        mejor_cv = (auc_cv, c)

modelo_final = LogisticRegression(C=mejor_cv[1], max_iter=2000, random_state=223201053)
modelo_final.fit(X_ent, y_ent)
acc_pru = accuracy_score(y_pru, modelo_final.predict(X_pru))
auc_pru = roc_auc_score(y_pru, modelo_final.predict_proba(X_pru)[:, 1])

print(f"C elegido (todos corregidos): {mejor_cv[1]}")
print(f"Exactitud en prueba: {acc_pru:.3f} AUC en prueba: {auc_pru:.3f}")

c:\Users\Laser\Desktop\examen-intermedio\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [37] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\Laser\Desktop\examen-intermedio\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


C elegido (todos corregidos): 0.01
Exactitud en prueba: 0.710 AUC en prueba: 0.711


---
## (d) La versión honesta
Construye un modelo sin esos problemas: todo el preprocesamiento dentro de un `Pipeline`, el desbalance tratado solo con el entrenamiento, los hiperparámetros elegidos con validación cruzada en el entrenamiento y una sola evaluación en la prueba. Reporta el resultado con un intervalo de confianza (por ejemplo, con bootstrap de la prueba).

In [ ]:
# TODO: Construye aquí el Pipeline honesto y calcula el intervalo de confianza por bootstrap


---
## (e) El experimento del ruido
Con $n = 50$, $5{,}000$ atributos de ruido, $20 + 5b = 45$ atributos seleccionados y un clasificador que tú eliges, repite al menos 50 veces la validación cruzada con la selección hecha:
1. **(i) Antes** de partir en pliegues.
2. **(ii) Dentro** de cada partición (con `Pipeline`).

Grafica las dos distribuciones de la exactitud estimada.

In [ ]:
# TODO: Simulación de los 5,000 atributos de ruido y gráfica de ambas distribuciones
